## Topic: Introduction of Text Splitter

### Agenda
- 1. Introduction of Text Splitter

- 2. Why Do We Need Text Splitter?

- 3. split_text() vs split_documents()

- 4. Chunk Size & Overlap Tuning Guide

- 5. Complete Taxonomy of Splitters

- 4. Complete Summary 

### 1. Introduction of Text Splitter

- Definition:
    - Text Splitting is the process of breaking large chunks of text (like articles, PDFs, HTML pages, or books) into smaller, manageable pieces (chunks) that an LLM can handle effectively.


- In LangChain: 
    - A Text Splitter is a LangChain component that takes large Document objects (from loaders) and cuts them into smaller chunks that fit within the LLM's token limit and embedding model's context window.



In [ ]:
""" - WORKFLOW:  IN LANGCHAIN WITH RAG BASE APPLICATION

┌─────────────────────────────────────────────────────────────┐
│         TEXT SPLITTERS IN THE RAG PIPELINE                  │
│                                                             │
│  PDF/Docs ──► [LOADER] ──► [SPLITTER] ──► [EMBEDDINGS]      │
│  (Raw Files)    List[Doc]     Chunks         Vectors        │
│                 (too big!)    (just right!)  (searchable)   │
│                               ↑                             │
│                         THIS IS WHERE                       │
│                        SPLITTERS WORK                       │
└─────────────────────────────────────────────────────────────┘


"""

### 2.Why Do We Need Text Splitter?

In [ ]:
"""     - Problem 1: Token Limits

┌─────────────────────────────────────────────────────────────┐
│              THE TOKEN LIMIT PROBLEM                        │
│                                                             │
│  Every LLM and embedding model has a MAXIMUM context:       │
│                                                             │
│  Model                  │ Max Tokens  │ Approx Words        │
│  ───────────────────────┼─────────────┼──────────────       │
│  text-embedding-3-small │ 8,191       │ ~6,000 words        │
│  GPT-4o                 │ 128,000     │ ~96,000 words       │
│  GPT-4o-mini            │ 128,000     │ ~96,000 words       │
│  Claude 3.5 Sonnet      │ 200,000     │ ~150,000 words      │
│                                                             │
│  A single PDF page can have 500-1000 words.                 │
│  A 200-page PDF = 100,000-200,000 words.                    │
│                                                             │
│  -  You CANNOT embed the entire PDF in one call!            │
│  -  You CANNOT send the entire PDF to the LLM!              │
│  -  You MUST split it into smaller chunks first.            │
└─────────────────────────────────────────────────────────────┘


"""

In [ ]:
"""   - Problem 2: Retrieval Quality / Embedding efficiency / Cost

- WHY NOT JUST SEND THE WHOLE DOCUMENT?

- Even if the LLM COULD handle 200K tokens, it SHOULDN'T:

1. RELEVANCE DILUTION
    - A 200-page document contains 1000+ topics.
    - When you ask "What is the refund policy?"(to specific topic), the LLM has to  search through 200 pages of irrelevant content.
    
    → With chunks: Only the 2-3 chunks about refunds are retrieved.

2. COST EXPLOSION
    - LLMs charge per token. Sending 200K tokens for every question 
    - costs 100x more than sending 2K tokens of relevant chunks.

3. LLM CONFUSION
    - Studies show LLMs perform WORSE with very long contexts.
    - They "forget" information in the middle of long documents (the "Lost in the Middle" problem).

4. SEARCH PRECISION
    - Vector search works best with focused, topic-specific chunks.
    - A chunk about "refund policy" has a clear embedding.
    - A 200-page document has a muddled, averaged embedding.

"""

- Key Note of Why Do We Need Chunking/splitting?

    - 1. Overcoming model limitation:
        - Many embedding models and language models have maximum input size constraints. Splitting allows us to process documents that would otherwise exceed these limit.

    - 2. Downstream tasks
        - Text splitting improves nearly every LLM powered task.
        - such as
            - Embedding task: short chunks yield more accurate vector.
            - Semantic search: Search results point to focused into, not noise.
            - Summarization: Prevents hallucination and topic drift.

    - 3. Optimizing computational resources:
        - Working with small chunks of text can be more memory-efficient and allow for better parallelization of processing tasks.


- The goal is:
    - Create chunks that are small enough for efficient retrieval but large enough to preserve meaningful context.

In [ ]:
""" 
    - Syntax of Text-Splitter

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(text)


- Key Note:
    - 1. Chunk Size:
        - This controls approximately how large each chunk can be, depending on the splitter and its length function.

        - For character-based splitters, it generally refers to characters.
        - For token-based splitters, it refers to tokens

    - 2. chunk_overlap: 
        - Without chunk_overlap
            - The relationship between the two pieces may become weaker.

        - With chunk_overlap:
            - The second chunk retains some surrounding context.
"""

### 3. split_text() vs split_documents()

In [ ]:
"""   split_text() vs split_documents()

- 1. split_text():
    - Use when we have plain text.

    - Example:
        - chunks = text_splitter.split_text(text)

    - Output:
        - List[str]
        - ["chunk 1", "chunk 2", "chunk 3"]

- 2. split_documents():

    - Use when we already have LangChain Document objects

    - Example:
        - chunks = text_splitter.split_documents(docs)

    - Output:
        - List[Document]
        - ["chunk Docs1", "chunk Docs2", "chunk Docs3"]

    - This is especially useful for RAG because metadata can be retained.
"""

###  4. Chunk Size & Overlap Tuning Guide

In [ ]:
"""   - How Splitting Works Internally

┌─────────────────────────────────────────────────────────────┐
│              SPLITTING ALGORITHM                             │
│                                                             │
│  INPUT: "A very long document with many paragraphs..."      │
│  PARAMS: chunk_size=100, chunk_overlap=20                   │
│                                                             │
│  STEP 1: Start at the beginning                             │
│          |================================================| │
│          ^                                                  │
│          pos=0                                              │
│                                                             │
│  STEP 2: Take chunk_size characters                         │
│          |==========|                                       │
│          pos=0      pos=100                                 │
│                                                             │
│  STEP 3: Try to split at the BEST separator                 │
│          (paragraph > sentence > word > character)          │
│          |========|                                         │
│          pos=0    pos=85 (split at last period)             │
│                                                             │
│  STEP 4: Move forward by (chunk_size - chunk_overlap)       │
│          New position = 85 - 20 = 65                        │
│                                                             │
│  STEP 5: Repeat from new position                           │
│                   |==========|                              │
│                   pos=65     pos=165                        │
│                                                             │
│  STEP 6: Continue until end of document                     │
│                                                             │
│  OUTPUT: [chunk1, chunk2, chunk3, ...]                      │
└─────────────────────────────────────────────────────────────┘



"""

In [ ]:
"""  - chunk_size and chunk_overlap — The Heart of Every Splitter

┌─────────────────────────────────────────────────────────────┐
│              chunk_size & chunk_overlap                     │
│                                                             │
│  chunk_size: The MAXIMUM size of each chunk                 │
│  ─────────────────────────────────────                      │
│  Measured in: characters (default) or tokens                │
│  Example: chunk_size=1000 → each chunk ≤ 1000 chars         │
│                                                             │
│  chunk_overlap: How much adjacent chunks OVERLAP            │
│  ─────────────────────────────────────                      │
│  Measured in: same unit as chunk_size                       │
│  Example: chunk_overlap=200 → 200 chars shared between      │
│           consecutive chunks                                │
│                                                             │
│  VISUAL:                                                    │
│  Original Text:                                             │
│  |======================================================|   │
│                                                             │
│  chunk_size=1000, chunk_overlap=200:                        │
│  Chunk 1: |==========|                                      │
│  Chunk 2:       |==========|    ← 200 chars overlap         │
│  Chunk 3:            |==========|                           │
│  Chunk 4:                 |==========|                      │
│                                                             │
│  WHY OVERLAP?                                               │
│  Without overlap, a sentence might get cut in half:         │
│  Chunk 1 ends: "The refund policy states that"              │
│  Chunk 2 starts: "customers can return within 30 days"      │
│  Neither chunk makes sense alone!                           │
│                                                             │
│  With overlap, both chunks contain the full sentence:       │
│  Chunk 1: "...The refund policy states that customers"      │
│  Chunk 2: "policy states that customers can return..."      │
│  Both chunks are self-contained!                            │
└─────────────────────────────────────────────────────────────┘

"""

In [ ]:
""" 
SMALL CHUNKS (200-500 chars)
────────────────────────────
 -  Precise retrieval (finds exact answers)
 -  Low token cost per query
 -  Better for factual Q&A
 -  Less context per chunk
 -  More chunks to embed (higher upfront cost)
 -  May lose important surrounding context

MEDIUM CHUNKS (800-1500 chars)  SWEET SPOT
─────────────────────────────────────────────
 -  Good balance of precision and context
 -  Works well for most RAG applications
 -  Reasonable embedding costs
 -  May include some irrelevant text

LARGE CHUNKS (2000-4000 chars)
──────────────────────────────
 -  Rich context per chunk
 -  Better for summarization tasks
 -  Fewer chunks to manage
 -  Noisy retrieval (too much irrelevant text)
 -  Higher token cost per query
 -  "Lost in the middle" problem

"""

In [ ]:
"""             - Chunk Size Trade-off   
    ==================================================


                 Chunk Size
                     │
        ┌────────────┴────────────┐
        ▼                         ▼
     Smaller                    Larger
        │                         │
        ▼                         ▼
 More precise                 More context
 retrieval                    per chunk
        │                         │
        ▼                         ▼
 Less context                 Less precise



            - Chunk Overlap Trade-off
    ==================================================

        Low overlap
            ↓
        Less duplicated content
            ↓
        Lower storage/token overhead


        High overlap
            ↓
        More contextual continuity
            ↓
        More duplicated content
            ↓
        Higher storage/token overhead  
        
"""

In [ ]:
"""   # Carefully Chunk Size Selection and chunk_overlap

- For Bad Chunks
=========================
      Bad chunks
         ↓
      Bad embeddings
         ↓
      Bad retrieval
         ↓
      Wrong context
         ↓
      Poor answer

- Whereas for Good Chunks:
=========================

      Good chunks
         ↓
      Good embeddings
         ↓
      Better retrieval
         ↓
      Relevant context
         ↓
      Better answer


- Key Note:
   - RAG quality is not only about choosing a good LLM. The ingestion and chunking strategy can be equally important.

"""

### 4. Complete Taxonomy of Splitters

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│                  TEXT SPLITTER TAXONOMY                          │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  CHARACTER-BASED (Most Common)                             │  │
│  │  ├── RecursiveCharacterTextSplitter  → Best all-rounder    │  │
│  │  └── CharacterTextSplitter           → Single separator    │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  TOKEN-BASED                                               │  │
│  │  ├── TokenTextSplitter               → tiktoken-based      │  │
│  │  └── SentenceTransformersSplitter    → Sentence-aware      │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  STRUCTURE-AWARE                                           │  │
│  │  ├── MarkdownHeaderTextSplitter      → By # headers        │  │
│  │  ├── HTMLHeaderTextSplitter          → By <h1>-<h6> tags   │  │
│  │  └── RecursiveJsonSplitter           → By JSON structure   │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  CODE-AWARE                                                │  │
│  │  ├── PythonCodeTextSplitter          → Python functions    │  │
│  │  ├── JavaScriptTextSplitter          → JS functions        │  │
│  │  ├── TypeScriptTextSplitter          → TS functions        │  │
│  │  ├── JavaTextSplitter                → Java classes        │  │
│  │  └── Language-agnostic               → 15+ languages       │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  SEMANTIC (AI-Powered)                                     │  │
│  │  └── SemanticChunker                 → By meaning shift    │  │
│  └────────────────────────────────────────────────────────────┘  │
└──────────────────────────────────────────────────────────────────┘

"""

In [ ]:
"""         Choosing the Right Splitter

What type of content are you splitting?
│
├─ General text (articles, reports, books)
│   └─→ RecursiveCharacterTextSplitter 
│       chunk_size=1000, chunk_overlap=200
│
├─ Markdown documentation
│   └─→ MarkdownHeaderTextSplitter
│       Split by # headers for structure-aware chunks
│
├─ HTML web pages
│   └─→ HTMLHeaderTextSplitter
│       Split by <h1>-<h6> tags
│
├─ Source code
│   └─→ RecursiveCharacterTextSplitter.from_language()
│       Language-aware splitting at function/class boundaries
│
├─ Need precise token control
│   └─→ TokenTextSplitter
│       chunk_size in tokens, not characters
│
├─ Topical coherence matters most
│   └─→ SemanticChunker
│       AI-powered splitting by meaning
│       Slower and more expensive
│
└─ Simple known format (e.g., CSV rows)
    └─→ CharacterTextSplitter
        Single separator, fast and predictable


"""



### 5. Complete Summary

In [ ]:
"""
┌──────────────────────────────────────────────────────────────────┐
│                      TEXT SPLITTERS                              │
│                                                                  │
│  WHAT:  Cut large Documents into smaller chunks for embedding    │
│  WHY:   Token limits, retrieval quality, cost efficiency         │
│  WHERE: After Loaders, before Embeddings in the RAG pipeline     │
│                                                                  │
│  KEY PARAMS:                                                     │
│    chunk_size    → Max size per chunk (chars or tokens)          │
│    chunk_overlap → Shared text between adjacent chunks           │
│                                                                  │
│  TOP SPLITTERS:                                                  │
│    -  RecursiveCharacterTextSplitter → Best all-rounder          │
│    -  MarkdownHeaderTextSplitter     → Structure-aware           │
│    -  from_language(Language.PYTHON) → Code-aware                │
│    -  TokenTextSplitter              → Token-precise             │
│    -  SemanticChunker                → Meaning-based (AI)        │
│                                                                  │
│  DEFAULT RECOMMENDATION:                                         │
│    RecursiveCharacterTextSplitter(                               │
│        chunk_size=1000,                                          │
│        chunk_overlap=200,                                        │
│        separators=["\n\n", "\n", ". ", ", ", " ", ""]            │
│    )                                                             │
│                                                                  │
│  PIPELINE:                                                       │
│    [Loader] → [Splitter] → [Embeddings] → [Vector Store]         │
│                  ↑                                               │
│            THIS COMPONENT                                        │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "Start with RecursiveCharacterTextSplitter at 1000/200.         │
│   Tune based on retrieval quality, not guesswork."               │
└──────────────────────────────────────────────────────────────────┘


"""

In [ ]:
"""     -  RAG Roadmap in LangChain  
      ===================================

                 RAG
                  │
                  ▼
          ┌───────────────┐
          │ Data Sources  │
          └───────┬───────┘
                  ▼
        ┌──────────────────┐
        │ Document Loaders │ 
        └────────┬─────────┘
                 ▼
            Documents
                 │
                 ▼
        ┌──────────────────┐
        │  Text Splitters  │ 
        └────────┬─────────┘
                 ▼
              Chunks
                 │
                 ▼
        ┌──────────────────┐
        │    Embeddings    │
        └────────┬─────────┘
                 ▼
        Vector Database
                 │
                 ▼
             Retriever
                 │
                 ▼
USER Query ➔   LLM
                 │
                 ▼
              Answer


"""